# 07 - Temporal Smoothing (Plan Section 10)

Collapses each device's daily sequence of Section 9
(`ml.bus_matching_global_assignment`) assignments into
`(bus_id, device_id, from_date, to_date)` intervals.

**Scope, confirmed live**: of 1,437 devices with at least one assigned
day, 1,407 (97.9%) stick to exactly one bus all month -- nothing to
smooth. Only 30 devices (60 device-bus pairs) ever show more than one
bus, and 25 of those 60 pairs are a single day. Hand-inspecting several
showed the exact pattern the plan describes: a solid run of bus A, one
day of bus B, the same run of A resuming. At this scale a transparent,
inspectable rule-based smoother (`app/smoothing.py`) is a better fit
than the plan's suggested HMM/changepoint-detection machinery -- see
that module's docstring for the full reasoning and the two rules it
implements (isolated one-day-deviation correction, and gap-bridging
that falls out of interval construction for free).

Every rule was verified against the real hand-inspected sequences
before being trusted here -- see the correctness checks below.


In [1]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml" / "bus_matching_model" / "app"))

In [2]:
import time

import pandas as pd
import psycopg
import smoothing

from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Tables

`ml.bus_matching_intervals`: the Section 10 deliverable -- one row per
contiguous `(device_id, bus_id)` run. `ml.bus_matching_smoothing_corrections`:
an audit log of every isolated-deviation correction, so a cluster on
one device (a real data problem, not noise) stays visible rather than
silently smoothed away.


In [3]:
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_intervals;")
intervals_ddl = (
    "CREATE TABLE ml.bus_matching_intervals (\n"
    "    device_id    text NOT NULL,\n"
    "    bus_id       text NOT NULL,\n"
    "    from_date    date NOT NULL,\n"
    "    to_date      date NOT NULL,\n"
    "    n_days       integer NOT NULL,\n"
    "    n_corrected  integer NOT NULL,\n"
    "    PRIMARY KEY (device_id, from_date)\n"
    ");"
)
conn.execute(intervals_ddl)

conn.execute("DROP TABLE IF EXISTS ml.bus_matching_smoothing_corrections;")
corrections_ddl = (
    "CREATE TABLE ml.bus_matching_smoothing_corrections (\n"
    "    device_id  text NOT NULL,\n"
    "    date       date NOT NULL,\n"
    "    from_bus   text NOT NULL,\n"
    "    to_bus     text NOT NULL,\n"
    "    PRIMARY KEY (device_id, date)\n"
    ");"
)
conn.execute(corrections_ddl)
conn.commit()
print("ml.bus_matching_intervals and ml.bus_matching_smoothing_corrections created")

ml.bus_matching_intervals and ml.bus_matching_smoothing_corrections created


## Run

In [4]:
start = time.monotonic()
assigned = pd.read_sql(
    "SELECT bus_id, date, device_id FROM ml.bus_matching_global_assignment "
    "WHERE device_id IS NOT NULL;",
    conn,
)
print(f"loaded {len(assigned)} assigned bus-days in {time.monotonic() - start:.1f}s")

intervals, corrections = smoothing.smooth_all_devices(assigned)
print(f"{len(intervals)} intervals, {len(corrections)} isolated-deviation corrections")

loaded 30121 assigned bus-days in 0.1s
1492 intervals, 23 isolated-deviation corrections


/tmp/ipykernel_1036053/4090293572.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  assigned = pd.read_sql(


In [5]:
with (
    conn.cursor() as cur,
    cur.copy(
        "COPY ml.bus_matching_intervals "
        "(device_id, bus_id, from_date, to_date, n_days, n_corrected) FROM STDIN"
    ) as copy,
):
    for row in intervals.itertuples(index=False):
        copy.write_row(row)

with (
    conn.cursor() as cur,
    cur.copy(
        "COPY ml.bus_matching_smoothing_corrections "
        "(device_id, date, from_bus, to_bus) FROM STDIN"
    ) as copy,
):
    for row in corrections.itertuples(index=False):
        copy.write_row(row)

conn.commit()
print(f"wrote {len(intervals)} intervals, {len(corrections)} corrections")

wrote 1492 intervals, 23 corrections


## Sanity checks

In [6]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT count(*), count(DISTINCT device_id) FROM ml.bus_matching_intervals;"
    )
    print("total intervals / distinct devices:", cur.fetchone())

    cur.execute("SELECT sum(n_days), sum(n_corrected) FROM ml.bus_matching_intervals;")
    print("sum n_days / sum n_corrected:", cur.fetchone())

    cur.execute(
        "SELECT count(*) FROM ml.bus_matching_global_assignment "
        "WHERE device_id IS NOT NULL;"
    )
    print(
        "assigned bus-days in Section 9 output (should match sum n_days):",
        cur.fetchone(),
    )

    # No two intervals for the same device may overlap in [from_date, to_date].
    overlap_sql = (
        "SELECT count(*) FROM ml.bus_matching_intervals a\n"
        "JOIN ml.bus_matching_intervals b\n"
        "  ON a.device_id = b.device_id AND a.from_date < b.from_date\n"
        "WHERE a.to_date >= b.from_date;"
    )
    cur.execute(overlap_sql)
    print("overlapping same-device intervals (must be 0):", cur.fetchone())

    cur.execute(
        "SELECT count(*) FROM ml.bus_matching_intervals "
        "GROUP BY device_id HAVING count(*) > 1;"
    )
    print("devices with more than one interval:", len(cur.fetchall()))

    # Mirrors Section 9's own uniqueness check, at the interval level: a
    # bus should never have two different devices with overlapping
    # intervals either.
    bus_overlap_sql = (
        "SELECT count(*) FROM ml.bus_matching_intervals a\n"
        "JOIN ml.bus_matching_intervals b\n"
        "  ON a.bus_id = b.bus_id AND a.device_id < b.device_id\n"
        "  AND a.from_date <= b.to_date AND b.from_date <= a.to_date;"
    )
    cur.execute(bus_overlap_sql)
    print(
        "same bus, two different devices, overlapping intervals (must be 0):",
        cur.fetchone(),
    )

    cur.execute(
        "SELECT device_id, count(*) AS n FROM ml.bus_matching_smoothing_corrections "
        "GROUP BY device_id ORDER BY n DESC LIMIT 5;"
    )
    print("devices with the most corrections (clusters worth investigating):")
    for row in cur.fetchall():
        print(" ", row)

total intervals / distinct devices: (1492, 1437)
sum n_days / sum n_corrected: (30121, 23)
assigned bus-days in Section 9 output (should match sum n_days): (30121,)
overlapping same-device intervals (must be 0): (0,)
devices with more than one interval: 40
same bus, two different devices, overlapping intervals (must be 0): (0,)
devices with the most corrections (clusters worth investigating):
  ('ep1-428113517', 2)
  ('ep1-428104922', 2)
  ('ep1-428104229', 1)
  ('ep1-428106412', 1)
  ('ep1-428103792', 1)


In [7]:
multi_sql = (
    "SELECT device_id, bus_id, from_date, to_date, n_days, n_corrected\n"
    "FROM ml.bus_matching_intervals\n"
    "WHERE device_id IN (\n"
    "    SELECT device_id FROM ml.bus_matching_intervals\n"
    "    GROUP BY device_id HAVING count(*) > 1\n"
    ")\n"
    "ORDER BY device_id, from_date\n"
    "LIMIT 20;"
)
sample = pd.read_sql(multi_sql, conn)
print("sample multi-interval devices:")
sample

sample multi-interval devices:


/tmp/ipykernel_1036053/2880245444.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql(multi_sql, conn)


,device_id,bus_id,from_date,to_date,n_days,n_corrected
0,ep1-163140320,14212,2023-11-13,2023-11-13,1,0
1,ep1-163140320,14212,2023-11-27,2023-11-27,1,0
2,ep1-428103592,30711,2023-11-01,2023-11-23,12,0
3,ep1-428103592,30711,2023-11-27,2023-11-29,3,0
4,ep1-428103715,12225,2023-11-01,2023-11-01,1,0
5,ep1-428103715,30708,2023-11-02,2023-11-03,2,0
6,ep1-428103715,12225,2023-11-06,2023-11-20,12,1
7,ep1-428103743,35230,2023-11-01,2023-11-08,4,0
8,ep1-428103743,35230,2023-11-10,2023-11-20,6,0
9,ep1-428103743,35230,2023-11-22,2023-11-27,4,0
